## AuxTel Tracking tests

Attempt to repeat tracking issues in the day with closed dome.

In [1]:
import sys, time, os, asyncio

from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from lsst.ts import salobj
from lsst.ts.observatory.control.auxtel.atcs import ATCS
from lsst.ts.observatory.control.auxtel.latiss import LATISS
from astropy.time import Time, TimeDelta
from astropy.coordinates import AltAz, ICRS, EarthLocation, Angle, FK5
import astropy.units as u
from lsst.ts.observatory.control.utils import RotType

/opt/lsst/software/stack/conda/miniconda3-py38_4.9.2/envs/lsst-scipipe-0.4.1/lib/python3.8/site-packages/jose/backends/cryptography_backend.py:23: CryptographyDeprecationWarning: int_from_bytes is deprecated, use int.from_bytes instead
  from cryptography.utils import int_from_bytes, int_to_bytes


In [2]:
# Set Cerro Pachon location
location = EarthLocation.from_geodetic(lon=-70.747698*u.deg,
                                       lat=-30.244728*u.deg,
                                       height=2663.0*u.m)

In [3]:
# for tab completion to work in current notebook instance
%config IPCompleter.use_jedi = False

In [4]:
import logging
stream_handler = logging.StreamHandler(sys.stdout)
logger = logging.getLogger()
logger.addHandler(stream_handler)
logger.level = logging.DEBUG
# Make matplotlib less chatty
logging.getLogger("matplotlib").setLevel(logging.WARNING)

In [5]:
#get classes and start them
domain = salobj.Domain()
await asyncio.sleep(10) # This can be removed in the future...
atcs = ATCS(domain)
latiss = LATISS(domain)
await asyncio.gather(atcs.start_task, latiss.start_task)

atmcs: Adding all resources.
atptg: Adding all resources.
ataos: Adding all resources.
atpneumatics: Adding all resources.
athexapod: Adding all resources.
atdome: Adding all resources.
atdometrajectory: Adding all resources.
atcamera: Adding all resources.
atspectrograph: Adding all resources.
atheaderservice: Adding all resources.
atarchiver: Adding all resources.
Read historical data in 0.00 sec
Read 1 history items for RemoteEvent(ATArchiver, 0, authList)
Read 1 history items for RemoteEvent(ATArchiver, 0, errorCode)
Read 100 history items for RemoteEvent(ATArchiver, 0, heartbeat)
Read 100 history items for RemoteEvent(ATArchiver, 0, imageInOODS)
Read 100 history items for RemoteEvent(ATArchiver, 0, imageRetrievalForArchiving)
Read 1 history items for RemoteEvent(ATArchiver, 0, logLevel)
Read 1 history items for RemoteEvent(ATArchiver, 0, logMessage)
Read 1 history items for RemoteEvent(ATArchiver, 0, simulationMode)
Read 1 history items for RemoteEvent(ATArchiver, 0, softwareVersi

[[None, None, None, None, None, None, None], [None, None, None, None]]

timeAndDate DDS read queue is full (100 elements); data may be lost
trajectory DDS read queue is filling: 37 of 100 elements
loadCell DDS read queue is filling: 22 of 100 elements
torqueDemand DDS read queue is filling: 37 of 100 elements
mount_positions DDS read queue is filling: 75 of 100 elements
nasymth_m3_mountMotorEncoders DDS read queue is filling: 37 of 100 elements
mountStatus DDS read queue is full (100 elements); data may be lost
mount_Nasmyth_Encoders DDS read queue is filling: 38 of 100 elements
guidingAndOffsets DDS read queue is full (100 elements); data may be lost
mount_AzEl_Encoders DDS read queue is filling: 38 of 100 elements
currentTargetStatus DDS read queue is full (100 elements); data may be lost
mount_AzEl_Encoders python read queue is filling: 37 of 100 elements
measuredTorque DDS read queue is filling: 38 of 100 elements
measuredMotorVelocity DDS read queue is filling: 38 of 100 elements
azEl_mountMotorEncoders DDS read queue is filling: 38 of 100 elements


In [ ]:
# Already enabled Skip
#await atcs.enable()
#await latiss.enable()

In [ ]:
await latiss.rem.atcamera.cmd_start.set_start(timeout=30)

In [ ]:
await salobj.set_summary_state(latiss.rem.atcamera, salobj.State.ENABLED)

In [ ]:
await latiss.rem.atarchiver.cmd_start.set_start(timeout=30)

In [186]:
await salobj.set_summary_state(atcs.rem.atptg, salobj.State.ENABLED)

[<State.FAULT: 3>, <State.STANDBY: 5>, <State.DISABLED: 1>, <State.ENABLED: 2>]

In [187]:
await salobj.set_summary_state(atcs.rem.atmcs, salobj.State.ENABLED)

[<State.FAULT: 3>, <State.STANDBY: 5>, <State.DISABLED: 1>, <State.ENABLED: 2>]

In [ ]:
# Everything now enabled 3:17PM

In [6]:
# take event checking out the slew commands to test telescope only
# otherwise it'll wait for the dome before completing slew command
atcs.check.atdome = False
atcs.check.atdometrajectory = False

In [7]:
# turn on ATAOS corrections just to make sure the mirror is under air
tmp = await atcs.rem.ataos.cmd_enableCorrection.set_start(m1=True, hexapod=True, atspectrograph=False)

In [ ]:
# Ensure we're using Nasmyth 2 - no longer needed
#await atcs.rem.atptg.cmd_focusName.set_start(focus=3)

In [ ]:
atcs.rem.atptg.cmd_raDecTarget.set(azWrapStrategy=1)  # 1 does not unwrap, 0 unwraps

In [102]:
# point telescope to desired starting position
start_az=180
start_el=35
start_rot_pa=0
#await atcs.point_azel(start_az, start_el, rot_tel=start_rot_pa, wait_dome=False)

In [103]:
# get RA/DEC of current telescope Alt/Az position
az = Angle(start_az, unit=u.deg)
el = Angle(start_el, unit=u.deg)
print(f'orig az {az} and el {el}')
time_data = await atcs.rem.atptg.tel_timeAndDate.next(flush=True, timeout=2)
# This should be TAI and not UTC... so will be 37s off system clock seconds ??
curr_time_atptg = Time(time_data.mjd, format="mjd")
coord_frame_AltAz = AltAz(location=location, obstime=curr_time_atptg)
coord_frame_radec = ICRS()
coord_azel = AltAz(az=az, alt=el, location=location, obstime=curr_time_atptg)
ra_dec = coord_azel.transform_to(coord_frame_radec)
print('Current Position is: \n {}'.format(coord_azel))
print('Current Position is: \n {}'.format(ra_dec))


orig az 180.0 deg and el 35.0 deg
Current Position is: 
 <AltAz Coordinate (obstime=59281.9727561976, location=(1819093.56876225, -5208411.6827961, -3195180.61110659) m, pressure=0.0 hPa, temperature=0.0 deg_C, relative_humidity=0.0, obswl=1.0 micron): (az, alt) in deg
    (180., 35.)>
Current Position is: 
 <ICRS Coordinate: (ra, dec) in deg
    (87.43443538, -85.24637932)>


In [68]:
atcs.slew_icrs?

Signature:
atcs.slew_icrs(
    ra,
    dec,
    rot=0.0,
    rot_type=<RotType.SkyAuto: 1>,
    target_name='slew_icrs',
    dra=0.0,
    ddec=0.0,
    slew_timeout=240.0,
    stop_before_slew=True,
    wait_settle=True,
)
Docstring:
Slew the telescope and start tracking an Ra/Dec target in ICRS
coordinate frame.

Parameters
----------
ra : `float`, `str` or `astropy.coordinates.Angle`
    Target RA, either as a float (hour), a sexagesimal string
    (HH:MM:SS.S or HH MM SS.S) coordinates or
    `astropy.coordinates.Angle`.
dec : `float`, `str` or `astropy.coordinates.Angle`
    Target Dec, either as a float (deg), a sexagesimal string
    (DD:MM:SS.S or DD MM SS.S) coordinates or
    `astropy.coordinates.Angle`.
rot : `float`, `str` or `astropy.coordinates.Angle`
    Specify desired rotation angle. The value will have different
    meaning depending on the choince of `rot_type` parameter.
    Accepts float (deg), sexagesimal string (DD:MM:SS.S or DD MM SS.S)
    coordinates or `astrop

In [69]:
atcs.atptg?

Object `atcs.atptg` not found.


In [ ]:
print(str(ra_dec.ra), str(ra_dec.dec))

In [ ]:
await atcs.rem.atptg.cmd_pointLoadModel.set_start(pointingFile="/home/saluser/repos/ts_pointing_common/install/data/at_20200219_manual.mod")

In [108]:
# Slew to starting position and take an image
await atcs.slew_icrs(ra=str(ra_dec.ra), dec=str(ra_dec.dec), rot=0.0, rot_type=RotType.PhysicalSky,
                      slew_timeout=240., stop_before_slew=False, wait_settle=True)


#await asyncio.sleep(2)
# take a 60s dark
#for i in range(5):
#    await latiss.take_darks(60.0, 1)


Setting rotator physical position to -30.0 deg. Rotator will track sky.
Parallactic angle: -0.7533210499565145 | Sky Angle: 174.24679927684275
xml 7 compatibility mode: rotPA=174.24679927684275, rotFrame=1
Sending command
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
[Telescope] delta Alt = +003.488 deg; delta Az= -005.734 deg; delta N1 = +000.001 deg; delta N2 = -022.770 deg 
[Telescope] delta Alt = +000.259 deg; delta Az= -001.178 deg; delta N1 = +000.000 deg; delta N2 = -021.428 deg 
atmcs not in <State.ENABLED: 2>: <State.FAULT: 3>


RuntimeError: atmcs state is <State.FAULT: 3>, expected <State.ENABLED: 2>

In [188]:
await atcs.point_azel(180, 35, rot_tel=0.0, wait_dome=False)

Sending command
Stop tracking.
Unknown tracking state: 10
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
[Telescope] delta Alt = -000.225 deg; delta Az= +000.003 deg; delta N1 = +000.000 deg; delta N2 = -033.937 deg 
[Telescope] delta Alt = +000.000 deg; delta Az= +000.001 deg; delta N1 = +000.000 deg; delta N2 = -028.064 deg 
[Telescope] delta Alt = +000.000 deg; delta Az= +000.000 deg; delta N1 = +000.000 deg; delta N2 = -024.130 deg 
[Telescope] delta Alt = +000.000 deg; delta Az= -000.000 deg; delta N1 = +000.000 deg; delta N2 = -018.379 deg 
[Telescope] delta Alt = +000.000 deg; delta Az= -000.000 deg; delta N1 = +000.000 deg; delta N2 = -014.746 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= -000.000 deg; delta N1 = +000.000 deg; delta N2 = -011.372 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= -000.000 deg; delta N1 = +000

In [ ]:
# now loop from elevation of +80 down to +20 in 5 degree increments
# Should be images 2 - 14
for i in range(13):
    start_el = 80 - i * 5
    # point telescope to desired starting position
    await atcs.point_azel(start_az, start_el, rot_tel=start_rot_pa, wait_dome=False)
    # get RA/DEC of current telescope Alt/Az position
    az = Angle(start_az, unit=u.deg)
    el = Angle(start_el, unit=u.deg)
    print(f'orig az {az} and el {el}')
    time_data = await atcs.rem.atptg.tel_timeAndDate.next(flush=True, timeout=2)
    # This should be TAI and not UTC... so will be 37s off system clock seconds ??
    curr_time_atptg = Time(time_data.mjd, format="mjd")
    coord_frame_AltAz = AltAz(location=location, obstime=curr_time_atptg)
    coord_frame_radec = ICRS()
    coord_azel = AltAz(az=az, alt=el, location=location, obstime=curr_time_atptg)
    ra_dec = coord_azel.transform_to(coord_frame_radec)
    print('Current Position is: \n {}'.format(coord_azel))
    print('Current Position is: \n {}'.format(ra_dec))
    # Slew to starting position and take an image to check headers and tracking
    await atcs.slew_icrs(ra=str(ra_dec.ra), dec=str(ra_dec.dec), rot=0.0,
                          slew_timeout=240., stop_before_slew=False, wait_settle=True)
    await asyncio.sleep(2)
    # take a 60s dark
    await latiss.take_darks(60.0, 1)
    


In [ ]:
# That took about 25 minutes.  So repeat 2 more times
# now loop from elevation of +80 down to +20 in 5 degree increments
# Should be images 15 - 41
for repeat in range(5):
    for i in range(13):
        start_el = 80 - i * 5
        # point telescope to desired starting position
        await atcs.point_azel(start_az, start_el, rot_tel=start_rot_pa, wait_dome=False)
        # get RA/DEC of current telescope Alt/Az position
        az = Angle(start_az, unit=u.deg)
        el = Angle(start_el, unit=u.deg)
        print(f'orig az {az} and el {el}')
        time_data = await atcs.rem.atptg.tel_timeAndDate.next(flush=True, timeout=2)
        # This should be TAI and not UTC... so will be 37s off system clock seconds ??
        curr_time_atptg = Time(time_data.mjd, format="mjd")
        coord_frame_AltAz = AltAz(location=location, obstime=curr_time_atptg)
        coord_frame_radec = ICRS()
        coord_azel = AltAz(az=az, alt=el, location=location, obstime=curr_time_atptg)
        ra_dec = coord_azel.transform_to(coord_frame_radec)
        print('Current Position is: \n {}'.format(coord_azel))
        print('Current Position is: \n {}'.format(ra_dec))
        # Slew to starting position and take an image to check headers and tracking
        await atcs.slew_icrs(ra=str(ra_dec.ra), dec=str(ra_dec.dec), rot=0.0,
                              slew_timeout=240., stop_before_slew=False, wait_settle=True)
        await asyncio.sleep(2)
        # take a 60s dark
        await latiss.take_darks(60.0, 1)
    


In [ ]:
# Repeat 3 more times at 3 different azimuths
# now loop from elevation of +80 down to +20 in 5 degree increments
# Should be images 42-
for repeat in range(3):
    start_az = 180 - 30 * repeat
    for i in range(13):
        start_el = 80 - i * 5
        # point telescope to desired starting position
        await atcs.point_azel(start_az, start_el, rot_tel=start_rot_pa, wait_dome=False)
        # get RA/DEC of current telescope Alt/Az position
        az = Angle(start_az, unit=u.deg)
        el = Angle(start_el, unit=u.deg)
        print(f'orig az {az} and el {el}')
        time_data = await atcs.rem.atptg.tel_timeAndDate.next(flush=True, timeout=2)
        # This should be TAI and not UTC... so will be 37s off system clock seconds ??
        curr_time_atptg = Time(time_data.mjd, format="mjd")
        coord_frame_AltAz = AltAz(location=location, obstime=curr_time_atptg)
        coord_frame_radec = ICRS()
        coord_azel = AltAz(az=az, alt=el, location=location, obstime=curr_time_atptg)
        ra_dec = coord_azel.transform_to(coord_frame_radec)
        print('Current Position is: \n {}'.format(coord_azel))
        print('Current Position is: \n {}'.format(ra_dec))
        # Slew to starting position and take an image to check headers and tracking
        await atcs.slew_icrs(ra=str(ra_dec.ra), dec=str(ra_dec.dec), rot=0.0,
                              slew_timeout=240., stop_before_slew=False, wait_settle=True)
        await asyncio.sleep(2)
        # take a 60s dark
        await latiss.take_darks(60.0, 1)
    


In [189]:
# For shutdown of system
await atcs.stop_tracking()

Stop tracking.
Unknown tracking state: 9
Unknown tracking state: 10
In Position: True.


In [190]:
# turn off corrections
tmp = await atcs.rem.ataos.cmd_disableCorrection.set_start(m1=True, hexapod=True, atspectrograph=True)

In [192]:
# take event checking out the slew commands to test telescope only
# otherwise it'll wait for the dome before completing slew command
atcs.check.atdome = True
atcs.check.atdometrajectory = True

In [193]:
# Putting everything back in standby.
# Failed as it usually does
await atcs.standby()

[atmcs]::[<State.STANDBY: 5>]
[atptg]::[<State.STANDBY: 5>]
[ataos]::[<State.STANDBY: 5>]
[atpneumatics]::[<State.STANDBY: 5>]
[athexapod]::[<State.STANDBY: 5>]
[atdome]::[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]
[atdometrajectory]::[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]
All components in <State.STANDBY: 5>.


In [ ]:
await atcs.rem.atdome.cmd_start.set_start(settingsToApply="test", timeout=30)

In [ ]:
await salobj.set_summary_state(atcs.rem.atdome, salobj.State.STANDBY, settingsToApply="test")

In [194]:
await salobj.set_summary_state(latiss.rem.atspectrograph, salobj.State.STANDBY)
await salobj.set_summary_state(latiss.rem.atcamera, salobj.State.STANDBY)
await salobj.set_summary_state(latiss.rem.atheaderservice, salobj.State.STANDBY)
await salobj.set_summary_state(latiss.rem.atarchiver, salobj.State.STANDBY)

[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]

In [ ]:
# Everything back in standby